# Photonic transient: split paths with propagation delay

This example launches one optical step into a 1×2 splitter and routes the outputs through waveguides of different lengths. The longer arm arrives later and experiences more propagation loss.

Circulax simulates the complex optical envelope rather than the hundreds-of-terahertz carrier. The explicit-delay waveguide preserves group delay $\tau=L n_g/c$ and field transmission $T=10^{-\alpha L/20}\exp(-j\phi)$. Each OpticalDelayLine receives its own interpolated history, so instances of different lengths retain different delays.

In [ ]:
import diffrax
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from circulax import compile_circuit
from circulax.components.electronic import Resistor
from circulax.components.photonic import OpticalDelayLine, OpticalSourcePulse, Splitter

jax.config.update("jax_enable_x64", True)
plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True, "figure.facecolor": "white"})

## Circuit and expected path properties

Both arms have the same propagation loss per centimetre; only their lengths differ. Matched $1\,\Omega$ optical-envelope loads let the 50/50 splitter feed both paths without reflections.

In [ ]:
c_um_per_s = 2.99792458e14
source_power, source_delay, source_rise = 1.0, 0.30e-9, 0.025e-9
split_ratio, n_group, loss_dB_cm = 0.5, 4.0, 3.0
short_length_um, long_length_um = 5_000.0, 15_000.0

def path_properties(length_um):
    delay = length_um * n_group / c_um_per_s
    loss_dB = loss_dB_cm * length_um / 10_000.0
    field_transmission = 10.0 ** (-loss_dB / 20.0)
    return delay, loss_dB, field_transmission

tau_short, loss_short_dB, transmission_short = path_properties(short_length_um)
tau_long, loss_long_dB, transmission_long = path_properties(long_length_um)

print(f"Short arm: {short_length_um / 1000:.1f} mm, {tau_short * 1e12:.1f} ps delay, {loss_short_dB:.1f} dB loss")
print(f"Long arm:  {long_length_um / 1000:.1f} mm, {tau_long * 1e12:.1f} ps delay, {loss_long_dB:.1f} dB loss")
print(f"Differential delay: {(tau_long - tau_short) * 1e12:.1f} ps")

In [ ]:
models = {
    "source": OpticalSourcePulse,
    "splitter": Splitter,
    "waveguide": OpticalDelayLine,
    "resistor": Resistor,
    "ground": lambda: 0,
}

netlist = {
    "instances": {
        "GND": {"component": "ground"},
        "SRC": {"component": "source", "settings": {
            "power": source_power, "delay": source_delay, "rise": source_rise,
        }},
        "SPLIT": {"component": "splitter", "settings": {"split_ratio": split_ratio}},
        "WG_SHORT": {"component": "waveguide", "settings": {
            "length_um": short_length_um, "loss_dB_cm": loss_dB_cm, "n_group": n_group,
        }},
        "WG_LONG": {"component": "waveguide", "settings": {
            "length_um": long_length_um, "loss_dB_cm": loss_dB_cm, "n_group": n_group,
        }},
        "TERM_SPLIT_SHORT": {"component": "resistor", "settings": {"R": 1.0}},
        "TERM_SPLIT_LONG": {"component": "resistor", "settings": {"R": 1.0}},
        "LOAD_SHORT": {"component": "resistor", "settings": {"R": 1.0}},
        "LOAD_LONG": {"component": "resistor", "settings": {"R": 1.0}},
    },
    "connections": {
        "GND,p1": (
            "SRC,p2", "TERM_SPLIT_SHORT,p2", "TERM_SPLIT_LONG,p2",
            "LOAD_SHORT,p2", "LOAD_LONG,p2",
        ),
        "SRC,p1": "SPLIT,p1",
        "SPLIT,p2": ("WG_SHORT,p1", "TERM_SPLIT_SHORT,p1"),
        "SPLIT,p3": ("WG_LONG,p1", "TERM_SPLIT_LONG,p1"),
        "WG_SHORT,p2": "LOAD_SHORT,p1",
        "WG_LONG,p2": "LOAD_LONG,p1",
    },
    "ports": {
        "input": "SRC,p1",
        "out_short": "WG_SHORT,p2",
        "out_long": "WG_LONG,p2",
    },
}

circuit = compile_circuit(netlist, models, is_complex=True, backend="dense")
y_dc = circuit.dc()
print(f"System size: {circuit.sys_size} complex unknowns")

## Transient simulation

The analytical references use only the configured split ratio, path attenuation, and group delay. Comparing envelope magnitudes removes carrier phase while retaining arrival time and loss.

In [ ]:
sample_times = jnp.linspace(0.0, 0.8e-9, 801)
solution = circuit.transient(
    t0=0.0,
    t1=float(sample_times[-1]),
    dt0=0.5e-12,
    y0=y_dc,
    saveat=diffrax.SaveAt(ts=sample_times),
    stepsize_controller=diffrax.PIDController(rtol=1e-5, atol=1e-7),
    max_steps=20_000,
    throw=True,
)

field_in = circuit.port(solution.ys, "input")
field_short = circuit.port(solution.ys, "out_short")
field_long = circuit.port(solution.ys, "out_long")
expected_short = jnp.sqrt(split_ratio) * transmission_short * jax.nn.sigmoid(
    (sample_times - source_delay - tau_short) / source_rise
)
expected_long = jnp.sqrt(1.0 - split_ratio) * transmission_long * jax.nn.sigmoid(
    (sample_times - source_delay - tau_long) / source_rise
)

short_error = float(jnp.max(jnp.abs(jnp.abs(field_short) - expected_short)))
long_error = float(jnp.max(jnp.abs(jnp.abs(field_long) - expected_long)))
print(f"Maximum short-arm envelope error: {short_error:.2e}")
print(f"Maximum long-arm envelope error:  {long_error:.2e}")
assert short_error < 5e-4
assert long_error < 5e-4

In [ ]:
time_ns = np.asarray(sample_times) * 1e9
fig, (ax_field, ax_power) = plt.subplots(1, 2, figsize=(12, 4))

ax_field.plot(time_ns, np.abs(field_in), color="0.35", lw=2, label="input")
ax_field.plot(time_ns, np.abs(field_short), color="C0", lw=2, label="5 mm arm")
ax_field.plot(time_ns, np.abs(field_long), color="C1", lw=2, label="15 mm arm")
ax_field.plot(time_ns, expected_short, "k--", lw=1, alpha=0.65, label="analytic shifts")
ax_field.plot(time_ns, expected_long, "k--", lw=1, alpha=0.65)
ax_field.axvline((source_delay + tau_short) * 1e9, color="C0", ls=":")
ax_field.axvline((source_delay + tau_long) * 1e9, color="C1", ls=":")
ax_field.set(xlabel="Time (ns)", ylabel=r"Envelope magnitude $|E|$",
             title="Different lengths produce different arrival times")
ax_field.legend(fontsize=9)

power_in = np.abs(field_in) ** 2
power_short = np.abs(field_short) ** 2
power_long = np.abs(field_long) ** 2
floor = 1e-12
ax_power.plot(time_ns, 10 * np.log10(np.maximum(power_in, floor)), color="0.35", lw=2, label="input")
ax_power.plot(time_ns, 10 * np.log10(np.maximum(power_short, floor)), color="C0", lw=2, label="5 mm arm")
ax_power.plot(time_ns, 10 * np.log10(np.maximum(power_long, floor)), color="C1", lw=2, label="15 mm arm")
ax_power.set(xlabel="Time (ns)", ylabel="Envelope power (dB, 1 W reference)",
             title="Longer propagation also produces more loss", ylim=(-35, 1))
ax_power.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Measured delay and loss

For a sigmoid field envelope, its centre is the half-amplitude crossing. We measure those crossings and the settled powers directly from the simulated traces.

In [ ]:
def half_amplitude_crossing(times, envelope):
    return np.interp(0.5 * envelope[-1], envelope, times)

arrival_short = half_amplitude_crossing(np.asarray(sample_times), np.abs(field_short))
arrival_long = half_amplitude_crossing(np.asarray(sample_times), np.abs(field_long))
measured_differential_delay = arrival_long - arrival_short

settled_short_power = float(jnp.abs(field_short[-1]) ** 2)
settled_long_power = float(jnp.abs(field_long[-1]) ** 2)
expected_short_power = split_ratio * source_power * 10.0 ** (-loss_short_dB / 10.0)
expected_long_power = (1.0 - split_ratio) * source_power * 10.0 ** (-loss_long_dB / 10.0)

print(f"Measured differential delay: {measured_differential_delay * 1e12:.1f} ps")
print(f"Expected differential delay: {(tau_long - tau_short) * 1e12:.1f} ps")
print(f"Short-arm settled power: {settled_short_power:.4f} W (expected {expected_short_power:.4f} W)")
print(f"Long-arm settled power:  {settled_long_power:.4f} W (expected {expected_long_power:.4f} W)")

assert abs(measured_differential_delay - (tau_long - tau_short)) < 1e-13
assert np.isclose(settled_short_power, expected_short_power, rtol=2e-4)
assert np.isclose(settled_long_power, expected_long_power, rtol=2e-4)

The two waveguides are instances of the same component, but each retains its own length-dependent history query. The result simultaneously shows the splitter ratio, accumulated path loss, and differential group delay.